# Fine-tuning para clasificación multiclase con Hugging Face

Vamos a usar el dataset `SetFit/amazon_reviews_multi_es`  , que cargaremos desde hugging face. 

Se trata de un conjunto de reseñas de productos de Amazon en español, que forma parte del dataset multilingüe *Amazon Reviews Multi*.

### 📊 Características principales

- **Tarea**: clasificación multiclase (5 clases)  
- **Clases**: valoraciones de 1 a 5 estrellas  
- **Idioma**: español  
- **Dominio**: reseñas de productos (e-commerce)  

### 🧾 Estructura de los datos

Cada ejemplo contiene:

- `text`: texto de la reseña  
- `label`: valoración numérica (0–4, correspondiente a 1–5 estrellas)  

### 📦 Tamaño

- Train: ~200k ejemplos  
- Test: ~5k ejemplos  

*(En este notebook usamos un subconjunto para facilitar la ejecución.)*


Modelo: `distilbert-base-multilingual-cased`

Objetivo: predecir la valoración de una reseña en español (5 clases).

## 1. Instalación

In [13]:
!pip install -q  datasets evaluate accelerate scikit-learn
!pip install -q --upgrade transformers

## 2. Imports

In [2]:
import numpy as np
from datasets import load_dataset
import evaluate
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

In [3]:
import transformers
print(transformers.__version__)

5.3.0


## 3. Cargar dataset

En este notebook no lo hacemos, pero deberíamos analizar el dataset para conocer la distribución de clases y el tamaño medio de sus textos (longitud en tokens).

In [4]:
dataset = load_dataset("SetFit/amazon_reviews_multi_es")
dataset

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
})

## 4. Subconjuntos para demo

In [14]:
train_ds = dataset["train"].shuffle(seed=42).select(range(4000))
val_ds = dataset["validation"].shuffle(seed=42).select(range(1000))
test_ds = dataset["test"].shuffle(seed=42).select(range(1000))

In [6]:
train_ds[0]

{'id': 'es_0614019',
 'text': 'La montarlo se rompió una rueda debido a materiales débiles, pero al arreglarla funciona correctamente.',
 'label': 2,
 'label_text': '2'}

## 5. Tokenización

In [16]:
model_checkpoint = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)
tokenized_test = test_ds.map(preprocess_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 6. Modelo

In [8]:
# Renombramos las labels para que sean más descriptivas (en el dataset original no están estos textos)
id2label = {0:"1_estrella",1:"2_estrellas",2:"3_estrellas",3:"4_estrellas",4:"5_estrellas"}
label2id = {v:k for k,v in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=5,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## 7. Métricas

In [9]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

## 8. Entrenamiento

In [10]:
training_args = TrainingArguments(
    output_dir="./ft_amazon_reviews_es",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## 9. Lanzar entrenamiento

In [11]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,2.831042,2.590068,0.424000,0.390535
2,2.450547,2.478627,0.443000,0.426870


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=250, training_loss=2.6855431518554687, metrics={'train_runtime': 105.8738, 'train_samples_per_second': 75.562, 'train_steps_per_second': 2.361, 'total_flos': 343183435016640.0, 'train_loss': 2.6855431518554687, 'epoch': 2.0})

In [23]:
print(trainer.state.log_history)

[{'loss': 3.144923095703125, 'grad_norm': 4.368912220001221, 'learning_rate': 1.6080000000000002e-05, 'epoch': 0.4, 'step': 50}, {'loss': 2.8310415649414065, 'grad_norm': 6.540842056274414, 'learning_rate': 1.2080000000000001e-05, 'epoch': 0.8, 'step': 100}, {'eval_loss': 2.5900683403015137, 'eval_accuracy': 0.424, 'eval_macro_f1': 0.39053519026594746, 'eval_runtime': 3.8287, 'eval_samples_per_second': 261.183, 'eval_steps_per_second': 8.358, 'epoch': 1.0, 'step': 125}, {'loss': 2.5554458618164064, 'grad_norm': 12.984880447387695, 'learning_rate': 8.08e-06, 'epoch': 1.2, 'step': 150}, {'loss': 2.4457583618164063, 'grad_norm': 14.514570236206055, 'learning_rate': 4.08e-06, 'epoch': 1.6, 'step': 200}, {'loss': 2.450546875, 'grad_norm': 11.458131790161133, 'learning_rate': 8e-08, 'epoch': 2.0, 'step': 250}, {'eval_loss': 2.4786272048950195, 'eval_accuracy': 0.443, 'eval_macro_f1': 0.426869920513124, 'eval_runtime': 3.7445, 'eval_samples_per_second': 267.059, 'eval_steps_per_second': 8.546

## 10. Evaluación sobre el conjunto test

In [17]:
# En vez de trainer.evaluate()
pred_output = trainer.predict(tokenized_test)
print(pred_output.metrics)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'test_loss': 2.5173256397247314, 'test_accuracy': 0.43, 'test_macro_f1': 0.41509767903973016, 'test_runtime': 3.8554, 'test_samples_per_second': 259.374, 'test_steps_per_second': 8.3}


## 11. Inferencia

In [25]:
import torch

examples = [
    "El producto llegó roto y además funciona fatal.",
    "Cumple su función, aunque la calidad es normal.",
    "Me encantó, volvería a comprarlo sin duda.",
]

inputs = tokenizer(examples, truncation=True, padding=True, return_tensors="pt")

device = model.device
inputs = {k: v.to(device) for k, v in inputs.items()}

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

pred_labels = outputs.logits.argmax(dim=-1).tolist()

for text, pred in zip(examples, pred_labels):
    print(text)
    print("Predicción:", id2label[int(pred)])
    print("-" * 60)

El producto llegó roto y además funciona fatal.
Predicción: 1_estrella
------------------------------------------------------------
Cumple su función, aunque la calidad es normal.
Predicción: 4_estrellas
------------------------------------------------------------
Me encantó, volvería a comprarlo sin duda.
Predicción: 5_estrellas
------------------------------------------------------------
